# 15. Animated GIFs: Water → Ice MD, NEB Paths, Crystal Orbits & 3D Isosurfaces

Static figures are great for papers and posters, while looping **animated GIFs** bring talks, lecture slides, and lab-meeting updates to life. `xkcd-matchem` provides three one-line animation helpers—all locking shared viewports across frames so atoms, hydrogen bonds, polyhedra, and orbital lobes move smoothly without zoom jitter:

- `xm.animate_trajectory()` — animates MD and NEB reaction trajectories from a `Trajectory` object or filename (with optional `show_hbonds=True` dashed hydrogen bonds and `ping_pong=True` forward-and-back looping).
- `xm.animate_orbit()` — sweeps a 360° camera orbit around any molecule, crystal, slab, or coordination polyhedra supercell.
- `xm.animate_isosurface()` — extracts a 3D orbital or charge-density isosurface once and orbits it in any of the 8 comic pen-and-ink shading modes (`solid`, `hatch`, `crosshatch`, `contour`, `cross_contour`, `stipple`, `scumble`, `outline`).

:::{note}
GIF export uses **Pillow** (`pip install "xkcd-matchem[animation] @ git+https://github.com/vzordillo/xkcd-matchem.git"`). Pillow is already pre-installed in Google Colab, so every cell below runs out of the box—no MLIP or GPU is needed because the bundled `water_to_ice.xyz` trajectory was pre-computed locally with MACE-MPA-0 (see `scripts/simulate_water_to_ice_mlip.py` to reproduce the simulation).
:::

In [ ]:
# Install xkcd-matchem automatically in Google Colab
import importlib.util


if importlib.util.find_spec("xkcd_matchem") is None:
    %pip install -q "git+https://github.com/vzordillo/xkcd-matchem.git"


## 1. Water → Ice I<sub>h</sub> MLIP MD Animation (`water_to_ice.xyz`)

The bundled `water_to_ice.xyz` dataset (32 H₂O / 96 atoms, 28 frames spanning 16.4 ps from 346 K down to a force-converged **0 K** Ice I<sub>h</sub> crystal at `q = 0.98`, `fmax = 0.02 eV/Å`) comes from an unconstrained **MACE-MPA-0** direct-coexistence Langevin MD crystallization + annealing + 0 K BFGS run generated locally via `scripts/simulate_water_to_ice_mlip.py`. Passing `show_hbonds=True` overlays dynamic dashed geometric `O–H···O` hydrogen bonds (`xm.infer_hbonds()`) so you can watch chaotic liquid bonds lock into hexagonal Ice I<sub>h</sub> cages in one call:

In [ ]:
import pathlib
import tempfile

from IPython.display import Image

import xkcd_matchem as xm


with tempfile.TemporaryDirectory() as tmpdir:
    gif_path = xm.animate_trajectory(
        "water_to_ice.xyz",
        pathlib.Path(tmpdir) / "water_to_ice.gif",
        fps=6,
        camera=xm.Camera(azimuth=-72.0, elevation=14.0, perspective=0.25),
        show_cell="clean",
        show_hbonds=True,
        figsize=(4.6, 5.2),
        dpi=95,
        title="Water -> Ice Ih",
    )
    md_gif_bytes = gif_path.read_bytes()

Image(data=md_gif_bytes, format="gif")


### Synchronized Water → Ice MD Storyboard (3D H-Bonds + Tetrahedral Order *q* + Δ*E*)

To make the phase transition unmistakable on a talk slide, pair the 3D water/ice coexistence box on the left with live traces of the **Errington–Debenedetti tetrahedral order parameter** (`traj.tetrahedral_order("O")`) and **exothermic potential energy release** (`traj.energy_profile(relative=True)`) on the right:

In [ ]:
import io

from matplotlib import pyplot as plt
import numpy as np
from PIL import Image as PilImage


water_md = xm.load_trajectory("water_to_ice.xyz")
times = water_md.times
rel_e = water_md.energy_profile(relative=True, unit="eV")
q_vals = water_md.tetrahedral_order("O")
hb_counts = [len(xm.infer_hbonds(frame)) for frame in water_md.frames]
print(
    f"Frames: {len(water_md)} | H-bonds: {hb_counts[0]} -> {hb_counts[-1]} "
    f"| q: {q_vals[0]:.2f} -> {q_vals[-1]:.2f} | dE: {rel_e[-1]:.2f} eV"
)

cam_ice = xm.Camera(azimuth=-72.0, elevation=14.0, perspective=0.25)
sw = xm.stroke_weights("classic")
t_end = float(times[-1])
ice_frames = []
frame_ids = sorted(set(range(0, len(water_md), 2)) | {len(water_md) - 1})
for idx in frame_ids:
    with xm.xkcd_style("classic"):
        fig = plt.figure(figsize=(9.4, 4.8), dpi=90)
        fig.patch.set_facecolor(xm.style.PAPER)
        fig.patch.set_sketch_params(0.0, 0.0, 0.0)
        gs = fig.add_gridspec(2, 2, width_ratios=[1.0, 1.15], hspace=0.45)
        ax_3d = fig.add_subplot(gs[:, 0])
        ax_q = fig.add_subplot(gs[0, 1])
        ax_e = fig.add_subplot(gs[1, 1])
        xm.render_structure_3d(
            water_md[idx],
            ax=ax_3d,
            camera=cam_ice,
            show_cell="clean",
            show_hbonds=True,
            radius_scale=0.88,
        )
        ax_3d.set_xlim(-6.2, 6.2)
        ax_3d.set_ylim(-9.0, 9.0)
        ax_3d.set_title(f"Water -> Ice Ih ({hb_counts[idx]} H-bonds)")
        ax_q.plot(times, q_vals, color="#0072B2", lw=sw.overlay, alpha=0.25)
        ax_q.plot(times[: idx + 1], q_vals[: idx + 1], color="#0072B2")
        ax_q.scatter(
            [times[idx]], [q_vals[idx]], s=80, color="#D55E00", clip_on=False
        )
        ax_q.set_xlim(-0.25, t_end + 0.35)
        ax_q.set_ylim(0.70, 1.04)
        ax_q.set_ylabel("Tetrahedral q")
        ax_q.set_title(f"t = {times[idx]:.1f} ps | q = {q_vals[idx]:.2f}")
        xm.style.apply_xkcd_spines(ax_q, preset="classic")
        ax_e.plot(times, rel_e, color="#009E73", lw=sw.overlay, alpha=0.25)
        ax_e.plot(times[: idx + 1], rel_e[: idx + 1], color="#009E73")
        ax_e.scatter(
            [times[idx]], [rel_e[idx]], s=80, color="#D55E00", clip_on=False
        )
        ax_e.set_xlim(-0.25, t_end + 0.35)
        ax_e.set_ylim(float(np.min(rel_e)) - 0.5, 0.5)
        ax_e.set_xlabel("Simulation Time (ps)")
        ax_e.set_ylabel("Delta E (eV)")
        xm.style.apply_xkcd_spines(ax_e, preset="classic")
        fig.subplots_adjust(left=0.04, right=0.97, top=0.90, bottom=0.12)
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=90, facecolor=fig.get_facecolor())
        plt.close(fig)
        buf.seek(0)
        ice_frames.append(PilImage.open(buf).convert("RGB"))

ice_sync_buf = io.BytesIO()
ice_frames[0].save(
    ice_sync_buf,
    format="GIF",
    save_all=True,
    append_images=ice_frames[1:],
    duration=220,
    loop=0,
    optimize=True,
)
Image(data=ice_sync_buf.getvalue(), format="gif")


## 2. Reaction Pathway GIFs (`sn2_neb.xyz`)

For Climbing-Image NEB trajectories, `ping_pong=True` reverses through interior images so the transition-state inversion loops smoothly without a jarring jump back to image 1 (and works identically for metallic MD trajectories like `cu_md.lammpstrj`):

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    gif_path = xm.animate_trajectory(
        "sn2_neb.xyz",
        pathlib.Path(tmpdir) / "sn2_neb.gif",
        fps=6,
        camera=xm.Camera(azimuth=-115.0, elevation=20.0, perspective=0.25),
        ping_pong=True,
        show_cell=False,
        dpi=95,
        title="SN2 Inversion",
    )
    neb_gif_bytes = gif_path.read_bytes()

Image(data=neb_gif_bytes, format="gif")


## 3. 360° Crystal & Coordination Polyhedra Orbit (`tio2_slab.poscar`)

`xm.animate_orbit()` automatically trims excess slab vacuum when `polyhedra=` is specified, tiles supercells, locks a constant 360° viewport, and renders an inline element legend plus optional crystallographic `a, b, c` triad:

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    gif_path = xm.animate_orbit(
        "tio2_slab.poscar",
        pathlib.Path(tmpdir) / "tio2_orbit.gif",
        n_frames=16,
        fps=8,
        supercell=(2, 2, 1),
        polyhedra=("Ti", "O"),
        crystal_axes="corner",
        dpi=95,
        title="TiO2 (110) Polyhedra Orbit",
    )
    orbit_gif_bytes = gif_path.read_bytes()

Image(data=orbit_gif_bytes, format="gif")


## 4. 360° 3D Orbital Isosurface Orbits (`water_homo.cube`)

`xm.animate_isosurface()` extracts the Marching Tetrahedra orbital meshes **once** and orbits them cleanly across all azimuth angles using any of the 8 comic pen-and-ink shading styles. Below we render a classic white-paper **cross-hatched** orbit first, followed by a dark chalkboard (`dark_chalk`) **stippled** slide animation:

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    gif_path = xm.animate_isosurface(
        "water_homo.cube",
        pathlib.Path(tmpdir) / "water_homo_crosshatch.gif",
        n_frames=14,
        fps=7,
        shade="crosshatch",
        dpi=95,
        title="Water HOMO Orbit (crosshatch)",
    )
    iso_light_bytes = gif_path.read_bytes()

Image(data=iso_light_bytes, format="gif")


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    gif_path = xm.animate_isosurface(
        "water_homo.cube",
        pathlib.Path(tmpdir) / "water_homo_dark.gif",
        n_frames=14,
        fps=7,
        shade="stipple",
        colors=("#56B4E9", "#F0A202"),
        alpha=0.38,
        preset="dark_chalk",
        dpi=95,
        title="Water HOMO Orbit (dark_chalk + stipple)",
    )
    iso_dark_bytes = gif_path.read_bytes()

Image(data=iso_dark_bytes, format="gif")


:::{tip}
**Reproducing the MLIP Water → Ice Trajectory Locally:** If you have `ase`, `torch`, and `mace-torch` installed, run `python scripts/simulate_water_to_ice_mlip.py --model medium-mpa-0 --device cuda` to regenerate `water_to_ice.xyz` from scratch. To save higher-resolution slide GIFs, swap the temporary directory for a local filename (e.g., `"water_to_ice.gif"`) with `n_frames=24`, `dpi=120`.
:::